# ABSA MBG — Analisis Sentimen Program Makan Bergizi Gratis
**Komparasi Multinomial Naïve Bayes vs LinearSVC**

Pipeline:
1. Instalasi & Import Library
2. Load Dataset
3. Preprocessing (Cleaning → Normalisasi → Segmentasi → Stopword → Stemming)
4. Pelabelan Otomatis (RoBERTa)
5. Identifikasi Aspek (Rule-Based)
6. Ekstraksi Fitur TF-IDF
7. Training Model (NB & SVM)
8. Evaluasi & Perbandingan
9. Simpan Model (Joblib)

---
## 1. Instalasi & Import Library

In [ ]:
# Install library yang tidak tersedia di Colab secara default
!pip install PySastrawi transformers accelerate openpyxl -q

In [ ]:
import pandas as pd
import numpy as np
import re
import string
import warnings
import joblib
import time
import requests
from io import BytesIO

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from wordcloud import WordCloud

import nltk
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

import torch
from transformers import pipeline as hf_pipeline

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

warnings.filterwarnings('ignore')
nltk.download('punkt', quiet=True)

# Cek GPU
device = 0 if torch.cuda.is_available() else -1
print(f"Device: {'GPU (CUDA)' if device == 0 else 'CPU'}")
print("Import selesai.")

---
## 2. Load Dataset

In [ ]:
from google.colab import files

print("Upload file CSV dataset:")
uploaded = files.upload()

In [ ]:
# Ambil nama file yang diupload
filename = list(uploaded.keys())[0]
df_raw = pd.read_csv(filename, encoding='utf-8')

# Pastikan kolom full_text tersedia
assert 'full_text' in df_raw.columns, "Kolom 'full_text' tidak ditemukan di CSV."

df_raw = df_raw[['full_text']].dropna().reset_index(drop=True)
df_raw['full_text'] = df_raw['full_text'].astype(str)

print(f"Dataset dimuat: {len(df_raw)} baris")
print(f"\nContoh data:")
df_raw.head()

---
## 3. Preprocessing

In [ ]:
# ── Inisialisasi Tools ────────────────────────────────────────
factory_stem = StemmerFactory()
stemmer = factory_stem.create_stemmer()

factory_stop = StopWordRemoverFactory()
sw_sastrawi = set(factory_stop.get_stop_words())
sw_custom = {
    "yg", "dg", "rt", "dgn", "ny", "d", "klo", "kalo", "amp",
    "biar", "bikin", "udah", "udh", "aja", "sih", "deh", "nih",
    "lah", "dong", "kan", "tuh", "mah", "wkwk", "haha", "hehe",
    "aku", "saya", "kamu", "dia", "kita", "kami", "mereka", "sama"
}
negation_words = {
    'tidak', 'tak', 'tiada', 'bukan', 'jangan',
    'belum', 'kurang', 'gak', 'ga', 'nggak', 'enggak'
}
final_stopwords = (sw_sastrawi | sw_custom) - negation_words

KONJUNGSI = r'\b(tetapi|namun|meskipun|tapi|sedangkan|cuman|cuma|sayangnya|padahal|walau|walaupun|pasalnya)\b'
KONJUNGSI_SET = {'tetapi', 'namun', 'meskipun', 'tapi', 'sedangkan',
                 'cuman', 'cuma', 'sayangnya', 'padahal', 'walau', 'walaupun', 'pasalnya'}

# ── Load Kamus Normalisasi ────────────────────────────────────
KAMUS_URL = "https://github.com/analysisdatasentiment/kamus_kata_baku/raw/main/kamuskatabaku.xlsx"
try:
    resp = requests.get(KAMUS_URL, timeout=15)
    xls = pd.read_excel(BytesIO(resp.content), engine='openpyxl')
    cols = xls.columns.tolist()
    norm_dict = dict(zip(
        xls[cols[0]].astype(str).str.lower(),
        xls[cols[1]].astype(str).str.lower()
    ))
    print(f"Kamus normalisasi: {len(norm_dict)} entri")
except Exception as e:
    norm_dict = {"yg": "yang", "gk": "tidak", "ga": "tidak", "gak": "tidak", "bgt": "banget"}
    print(f"Gagal load kamus ({e}), pakai fallback.")

print("Resources preprocessing siap.")

In [ ]:
# ── Fungsi Preprocessing ──────────────────────────────────────
def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r'@\w+', ' ', text)
    text = re.sub(r'#\w+', ' ', text)
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    return re.sub(r'\s+', ' ', text).strip()

def normalize_text(text: str) -> str:
    return ' '.join(norm_dict.get(w, w) for w in text.split())

def segmentasi_kalimat(text: str) -> list:
    parts = re.split(KONJUNGSI, text)
    return [s.strip() for s in parts
            if s.strip() and s.strip() not in KONJUNGSI_SET] or [text]

def stopword_and_stem(text: str) -> str:
    words = [w for w in text.split() if w not in final_stopwords]
    return stemmer.stem(' '.join(words))

def preprocess_pipeline(text: str) -> list:
    cleaned  = clean_text(text)
    normed   = normalize_text(cleaned)
    segments = segmentasi_kalimat(normed)
    result   = []
    for seg in segments:
        processed = stopword_and_stem(seg)
        if processed.strip():
            result.append(processed)
    return result

print("Fungsi preprocessing terdefinisi.")

# ── Contoh Trace Before-After ─────────────────────────────────
contoh = "makanan bergizi gratis ini enak banget tapi distribusinya sering telat ke sekolah!"
print(f"\nContoh trace preprocessing:")
print(f"  Asli     : {contoh}")
print(f"  Cleaned  : {clean_text(contoh)}")
print(f"  Normed   : {normalize_text(clean_text(contoh))}")
segs = segmentasi_kalimat(normalize_text(clean_text(contoh)))
print(f"  Segmen   : {segs}")
print(f"  Final    : {[stopword_and_stem(s) for s in segs]}")

In [ ]:
# ── Jalankan Preprocessing pada Seluruh Data ──────────────────
print("Menjalankan preprocessing...")
t0 = time.time()

# Hapus duplikat (anti-buzzer)
df = df_raw.drop_duplicates(subset=['full_text']).copy().reset_index(drop=True)
print(f"  Setelah hapus duplikat: {len(df)} baris (dihapus {len(df_raw)-len(df)})")

# Preprocessing + segmentasi + explode
df['doc_id']      = df.index
df['text_clean']  = df['full_text'].apply(clean_text)
df['text_norm']   = df['text_clean'].apply(normalize_text)
df['segmen_list'] = df['text_norm'].apply(segmentasi_kalimat)

df_exp = df.explode('segmen_list').dropna(subset=['segmen_list'])
df_exp['segment_raw'] = df_exp['segmen_list'].astype(str).str.strip()
df_exp = df_exp[df_exp['segment_raw'] != ''].copy()

# Stopword removal + stemming
df_exp['segment'] = df_exp['segment_raw'].apply(stopword_and_stem)
df_exp = df_exp[df_exp['segment'].str.strip() != ''].reset_index(drop=True)

elapsed = time.time() - t0
print(f"  Setelah segmentasi & stemming: {len(df_exp)} segmen")
print(f"  Waktu preprocessing: {elapsed:.1f} detik")
df_exp[['doc_id', 'full_text', 'segment']].head(10)

---
## 4. Pelabelan Otomatis dengan RoBERTa

In [ ]:
# Load model RoBERTa (GPU otomatis dipakai kalau tersedia)
print("Loading model RoBERTa...")
MODEL_NAME = "w11wo/indonesian-roberta-base-sentiment-classifier"

sentiment_classifier = hf_pipeline(
    "text-classification",
    model=MODEL_NAME,
    device=device,
    truncation=True,
    max_length=128
)
print(f"Model siap. Device: {'GPU' if device == 0 else 'CPU'}")

In [ ]:
# ── Fungsi Labeling ───────────────────────────────────────────
def label_to_indo(label: str) -> str:
    label = label.lower()
    if 'pos' in label: return 'Positif'
    if 'neg' in label: return 'Negatif'
    return 'Netral'

def batch_predict(texts: list, batch_size: int = 64) -> list:
    results = []
    total   = len(texts)
    for i in range(0, total, batch_size):
        batch  = texts[i:i+batch_size]
        preds  = sentiment_classifier(batch)
        labels = [label_to_indo(p['label']) for p in preds]
        results.extend(labels)
        if (i // batch_size) % 10 == 0:
            print(f"  Progress: {min(i+batch_size, total)}/{total} segmen", end='\r')
    return results

# ── Jalankan Labeling ─────────────────────────────────────────
# RoBERTa dijalankan pada segment_raw (sebelum stemming)
# supaya konteks kalimat tidak rusak saat inferensi
print("Menjalankan pelabelan RoBERTa...")
t0 = time.time()

texts_for_labeling = df_exp['segment_raw'].tolist()
df_exp['sentiment_label'] = batch_predict(texts_for_labeling)

elapsed = time.time() - t0
print(f"\nPelabelan selesai: {elapsed/60:.1f} menit")

# ── Distribusi ────────────────────────────────────────────────
dist = df_exp['sentiment_label'].value_counts()
pct  = df_exp['sentiment_label'].value_counts(normalize=True).mul(100).round(1)
print("\nDistribusi Sentimen:")
for label in dist.index:
    print(f"  {label}: {dist[label]} ({pct[label]}%)")

In [ ]:
# ── Visualisasi Distribusi Sentimen ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie chart
colors_map = {'Positif': '#2ecc71', 'Negatif': '#e74c3c', 'Netral': '#95a5a6'}
vals  = df_exp['sentiment_label'].value_counts()
cols  = [colors_map.get(l, '#3498db') for l in vals.index]
axes[0].pie(vals.values, labels=vals.index, autopct='%1.1f%%',
            colors=cols, startangle=90)
axes[0].set_title('Distribusi Sentimen (Semua Data)', fontweight='bold')

# Bar chart
sns.barplot(x=vals.index, y=vals.values, palette=cols, ax=axes[1], hue=vals.index, legend=False)
axes[1].set_title('Jumlah Segmen per Sentimen', fontweight='bold')
axes[1].set_ylabel('Jumlah Segmen')
for i, v in enumerate(vals.values):
    axes[1].text(i, v + 50, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('distribusi_sentimen.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot disimpan: distribusi_sentimen.png")

---
## 5. Identifikasi Aspek (Rule-Based)

In [ ]:
ASPEK_DICT = {
    'Kualitas': [
        'kualitas', 'bagus', 'jelek', 'enak', 'basi', 'gizi', 'susu',
        'menu', 'rasa', 'porsi', 'higienis', 'keracunan', 'sehat',
        'mentah', 'keras', 'hambar', 'ulat', 'lauk', 'sayur',
        'karbohidrat', 'protein', 'lemak', 'gula', 'ayam', 'telur',
        'kenyang', 'alergi', 'higienitas'
    ],
    'Layanan': [
        'layan', 'antri', 'ramah', 'lambat', 'cepat', 'bantu', 'saji',
        'distribusi', 'vendor', 'katering', 'sekolah', 'siswa', 'guru',
        'telat', 'molor', 'bocor', 'tepat waktu', 'pelosok', 'merata',
        'zonasi', 'umkm', 'kemasan', 'kotak', 'plastik'
    ],
    'Anggaran': [
        'harga', 'mahal', 'murah', 'biaya', 'bayar', 'anggar', 'boros',
        'korupsi', 'dana', 'apbn', 'pajak', 'potong', 'sunat', 'markup',
        'tender', 'proyek', 'apbd', 'defisit', 'utang', 'ekonomi',
        'alokasi', 'transparan'
    ],
}

def get_aspects(text: str) -> list:
    tokens = set(str(text).split())
    found  = [asp for asp, keys in ASPEK_DICT.items()
               if not tokens.isdisjoint(keys)]
    return found if found else ['Lainnya']

# Identifikasi aspek pada segmen yang sudah di-stem
df_exp['aspect_list'] = df_exp['segment'].apply(get_aspects)

# Statistik aspek
df_asp = df_exp.explode('aspect_list')
asp_counts = df_asp['aspect_list'].value_counts()
print("Distribusi Aspek:")
print(asp_counts.to_string())

In [ ]:
# ── Visualisasi Aspek ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart distribusi aspek
sns.barplot(x=asp_counts.index, y=asp_counts.values,
            palette='viridis', ax=axes[0], hue=asp_counts.index, legend=False)
axes[0].set_title('Distribusi Aspek (Semua Segmen)', fontweight='bold')
axes[0].set_ylabel('Jumlah Segmen')
for i, v in enumerate(asp_counts.values):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')

# Stacked bar: sentimen per aspek
df_asp_sent = df_asp[df_asp['aspect_list'] != 'Lainnya'].copy()
pivot = df_asp_sent.groupby(['aspect_list', 'sentiment_label']).size().unstack(fill_value=0)
pivot.plot(kind='bar', ax=axes[1], color=['#e74c3c', '#2ecc71', '#95a5a6'],
           edgecolor='white')
axes[1].set_title('Sentimen per Aspek', fontweight='bold')
axes[1].set_ylabel('Jumlah Segmen')
axes[1].set_xlabel('')
axes[1].legend(title='Sentimen')
plt.xticks(rotation=0)

plt.tight_layout()
plt.savefig('distribusi_aspek.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot disimpan: distribusi_aspek.png")

In [ ]:
# ── WordCloud per Sentimen ────────────────────────────────────
sentimen_list = ['Positif', 'Negatif']
cmap_list     = ['Greens', 'Reds']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, sent, cmap in zip(axes, sentimen_list, cmap_list):
    teks = " ".join(df_exp[df_exp['sentiment_label'] == sent]['segment'].astype(str))
    if teks.strip():
        wc = WordCloud(width=600, height=350, background_color='white',
                       colormap=cmap, max_words=80).generate(teks)
        ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'WordCloud — {sent}', fontweight='bold', fontsize=13)

plt.tight_layout()
plt.savefig('wordcloud_sentimen.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot disimpan: wordcloud_sentimen.png")

---
## 6. Ekstraksi Fitur TF-IDF

In [ ]:
# Hapus kelas Netral dari training (klasifikasi biner)
df_model = df_exp[df_exp['sentiment_label'] != 'Netral'].copy()
netral_count = len(df_exp) - len(df_model)
print(f"Segmen Netral dihapus dari training: {netral_count}")
print(f"Sisa untuk training: {len(df_model)} segmen")
print(f"Distribusi:\n{df_model['sentiment_label'].value_counts()}")

X = df_model['segment']
y = df_model['sentiment_label']

# Train-test split 80:20 stratified
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain: {len(X_train)} | Test: {len(X_test)}")

In [ ]:
# TF-IDF — fit HANYA pada data training
TFIDF_PARAMS = {
    'max_features': 3000,
    'ngram_range': (1, 2),
    'sublinear_tf': True,
    'min_df': 2,
    'max_df': 0.85
}

tfidf = TfidfVectorizer(**TFIDF_PARAMS)
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec  = tfidf.transform(X_test)

print(f"Vocabulary size: {len(tfidf.vocabulary_)}")
print(f"Shape training matrix: {X_train_vec.shape}")
print(f"Shape test matrix    : {X_test_vec.shape}")

# Top 10 kata bobot TF-IDF tertinggi
sum_tfidf = X_train_vec.sum(axis=0)
words_arr = tfidf.get_feature_names_out()
df_tfidf  = pd.DataFrame({'Kata': words_arr,
                           'Skor': np.asarray(sum_tfidf).flatten()})
df_tfidf  = df_tfidf.sort_values('Skor', ascending=False).head(10)
print(f"\nTop 10 Kata TF-IDF:")
print(df_tfidf.to_string(index=False))

---
## 7. Training Model

In [ ]:
# ── Multinomial Naive Bayes ───────────────────────────────────
print("Training Multinomial Naive Bayes...")
t0 = time.time()

nb_model = MultinomialNB(alpha=0.1)
nb_model.fit(X_train_vec, y_train)
y_pred_nb = nb_model.predict(X_test_vec)

print(f"  Waktu training NB: {time.time()-t0:.3f} detik")

# ── LinearSVC ─────────────────────────────────────────────────
print("Training LinearSVC...")
t0 = time.time()

svm_model = LinearSVC(random_state=42, max_iter=2000, C=1.0)
svm_model.fit(X_train_vec, y_train)
y_pred_svm = svm_model.predict(X_test_vec)

print(f"  Waktu training SVM: {time.time()-t0:.3f} detik")
print("\nTraining selesai.")

---
## 8. Evaluasi & Perbandingan

In [ ]:
# ── Metrik Global ─────────────────────────────────────────────
def get_metrics(y_true, y_pred, name):
    return {
        'Model'    : name,
        'Accuracy' : accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'Recall'   : recall_score(y_true, y_pred, average='weighted', zero_division=0),
        'F1-Score' : f1_score(y_true, y_pred, average='weighted', zero_division=0),
    }

metrics_nb  = get_metrics(y_test, y_pred_nb,  'Multinomial NB')
metrics_svm = get_metrics(y_test, y_pred_svm, 'LinearSVC')

df_metrics = pd.DataFrame([metrics_nb, metrics_svm]).set_index('Model')
print("=" * 55)
print("METRIK EVALUASI GLOBAL")
print("=" * 55)
print(df_metrics.to_string())

print("\nClassification Report — Multinomial NB:")
print(classification_report(y_test, y_pred_nb, zero_division=0))

print("Classification Report — LinearSVC:")
print(classification_report(y_test, y_pred_svm, zero_division=0))

In [ ]:
# ── Visualisasi Perbandingan & Confusion Matrix ───────────────
labels_cm = sorted(y_test.unique())
cm_nb     = confusion_matrix(y_test, y_pred_nb,  labels=labels_cm)
cm_svm    = confusion_matrix(y_test, y_pred_svm, labels=labels_cm)

fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig)

# Bar chart perbandingan metrik
ax_bar = fig.add_subplot(gs[0, :])
df_melt = df_metrics.reset_index().melt(
    id_vars='Model', var_name='Metrik', value_name='Skor'
)
sns.barplot(data=df_melt, x='Metrik', y='Skor', hue='Model',
            palette=['#3498db', '#e67e22'], ax=ax_bar)
ax_bar.set_ylim(0, 1.15)
ax_bar.set_title('Perbandingan Metrik Evaluasi — NB vs LinearSVC',
                 fontweight='bold', fontsize=13)
ax_bar.set_ylabel('Skor')
for p in ax_bar.patches:
    if p.get_height() > 0:
        ax_bar.annotate(f'{p.get_height():.4f}',
                        (p.get_x() + p.get_width()/2, p.get_height() + 0.01),
                        ha='center', fontsize=9)

# Confusion Matrix NB
ax_cm1 = fig.add_subplot(gs[1, 0])
sns.heatmap(cm_nb, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels_cm, yticklabels=labels_cm, ax=ax_cm1)
ax_cm1.set_title('Confusion Matrix — Multinomial NB', fontweight='bold')
ax_cm1.set_xlabel('Prediksi'); ax_cm1.set_ylabel('Aktual')

# Confusion Matrix SVM
ax_cm2 = fig.add_subplot(gs[1, 1])
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=labels_cm, yticklabels=labels_cm, ax=ax_cm2)
ax_cm2.set_title('Confusion Matrix — LinearSVC', fontweight='bold')
ax_cm2.set_xlabel('Prediksi'); ax_cm2.set_ylabel('Aktual')

plt.tight_layout()
plt.savefig('evaluasi_model.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot disimpan: evaluasi_model.png")

In [ ]:
# ── Evaluasi Per Aspek ────────────────────────────────────────
print("=" * 55)
print("EVALUASI PER ASPEK")
print("=" * 55)

# Gabungkan prediksi ke df_model
test_idx  = X_test.index
df_test   = df_model.loc[test_idx].copy()
df_test['pred_nb']  = y_pred_nb
df_test['pred_svm'] = y_pred_svm
df_test_exp = df_test.explode('aspect_list')

asp_results = []
for asp in df_test_exp['aspect_list'].unique():
    sub = df_test_exp[df_test_exp['aspect_list'] == asp]
    if len(sub) == 0:
        continue
    asp_results.append({
        'Aspek'          : asp,
        'Jumlah Segmen'  : len(sub),
        'Akurasi NB'     : accuracy_score(sub['sentiment_label'], sub['pred_nb']),
        'F1 NB'          : f1_score(sub['sentiment_label'], sub['pred_nb'],
                                    average='weighted', zero_division=0),
        'Akurasi SVM'    : accuracy_score(sub['sentiment_label'], sub['pred_svm']),
        'F1 SVM'         : f1_score(sub['sentiment_label'], sub['pred_svm'],
                                    average='weighted', zero_division=0),
    })

df_asp_eval = pd.DataFrame(asp_results).sort_values('Jumlah Segmen', ascending=False)
print(df_asp_eval.to_string(index=False))

# Visualisasi per aspek
df_asp_no_other = df_asp_eval[df_asp_eval['Aspek'] != 'Lainnya']
if not df_asp_no_other.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    df_asp_no_other.melt(
        id_vars='Aspek', value_vars=['Akurasi NB', 'Akurasi SVM'],
        var_name='Model', value_name='Akurasi'
    ).pipe(lambda d: sns.barplot(
        data=d, x='Aspek', y='Akurasi', hue='Model',
        palette=['#3498db', '#e67e22'], ax=ax
    ))
    ax.set_ylim(0, 1.15)
    ax.set_title('Perbandingan Akurasi per Aspek', fontweight='bold')
    for p in ax.patches:
        if p.get_height() > 0:
            ax.annotate(f'{p.get_height():.2f}',
                        (p.get_x() + p.get_width()/2, p.get_height() + 0.01),
                        ha='center', fontsize=9)
    plt.tight_layout()
    plt.savefig('evaluasi_per_aspek.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Plot disimpan: evaluasi_per_aspek.png")

---
## 9. Simpan Model (Joblib)

In [ ]:
# Simpan semua komponen yang dibutuhkan Streamlit
saved_data = {
    'model_nb'       : nb_model,
    'model_svm'      : svm_model,
    'vectorizer'     : tfidf,
    'y_test'         : y_test,
    'y_pred_nb'      : y_pred_nb,
    'y_pred_svm'     : y_pred_svm,
    'test_data_eval' : df_test,
    'metrics_nb'     : metrics_nb,
    'metrics_svm'    : metrics_svm,
}

joblib.dump(saved_data, 'saved_model_data.joblib')
print("Model disimpan: saved_model_data.joblib")

# Simpan juga hasil preprocessing untuk referensi
df_exp.to_csv('hasil_preprocessing_labeled.csv', index=False)
print("Dataset berlabel disimpan: hasil_preprocessing_labeled.csv")

In [ ]:
# Download semua file hasil ke komputer lokal
from google.colab import files

for fname in [
    'saved_model_data.joblib',
    'hasil_preprocessing_labeled.csv',
    'distribusi_sentimen.png',
    'distribusi_aspek.png',
    'wordcloud_sentimen.png',
    'evaluasi_model.png',
    'evaluasi_per_aspek.png',
]:
    try:
        files.download(fname)
        print(f"Downloaded: {fname}")
    except Exception as e:
        print(f"Gagal download {fname}: {e}")

---
## Ringkasan Hasil

In [ ]:
print("=" * 55)
print("RINGKASAN HASIL PENELITIAN")
print("=" * 55)
print(f"Total data mentah       : {len(df_raw)}")
print(f"Setelah hapus duplikat  : {len(df)}")
print(f"Total segmen            : {len(df_exp)}")
print(f"Segmen untuk training   : {len(df_model)}")
print(f"  - Data train (80%)    : {len(X_train)}")
print(f"  - Data test  (20%)    : {len(X_test)}")
print()
print("Distribusi Sentimen (setelah hapus Netral):")
print(df_model['sentiment_label'].value_counts().to_string())
print()
print("Hasil Evaluasi:")
print(df_metrics.to_string())
print()
best = 'LinearSVC' if metrics_svm['F1-Score'] >= metrics_nb['F1-Score'] else 'Multinomial NB'
print(f"Model terbaik berdasarkan F1-Score: {best}")